# v10 B300 — the sm_103 2×-exp softmax delta (T2)

B300 doubles transcendental/EX2 throughput (5 → **10.7 TeraExp/s**) vs B200 — the cleanest sm_103-vs-
sm_100 lever, and softmax is one exp per score. This notebook measures (1) **achieved EX2 throughput**
(prediction-vs-measured vs the 10.7 claim) via a compute-bound microkernel, and (2) the **softmax-term
share of the decode floor** (the roofline `t_mufu` vs `t_hbm`/`t_mma`) — to show how much the 2×-exp can
actually move per-CTA-bound M=1 decode (kickoff prediction: small, but a first-class measured number).

## 0. Dependencies + GPU

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU. FIX: rent a B200 (sm_100) or B300 (sm_103) on vast.ai — privileged/bare-metal for ncu.')

# matplotlib for the decisive plot (the one new dep vs other gates); numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    # Blackwell (B200 sm_100 / B300 sm_103) has no SASS in the cu124 wheel; nightly cu129 covers both.
    # (Skipped entirely if your image already ships a Blackwell-capable torch.)
    pip('--pre', 'torch', extra=('--index-url', 'https://download.pytorch.org/whl/nightly/cu129'))
    raise SystemExit('Installed Blackwell torch (nightly cu129). RESTART the kernel + re-run from the top.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. EX2 throughput microkernel (compute-bound) — measured vs the 10.7 TExp/s claim

In [ ]:
# A tiny inline kernel: each thread does `iters` dependent __expf in registers (no memory traffic),
# so wall-time / total-exps = achieved EX2 throughput. Compare to roofline exp_per_s (10.7 TExp/s).
import torch
from torch.utils.cpp_extension import load_inline
ex2 = load_inline(name='ex2_bench_mod', cpp_sources='', cuda_sources="#include <torch/extension.h>\n#include <cuda_runtime.h>\n__global__ void ex2_bench(float* out, int iters) {\n    // Compute-bound EX2: each thread does `iters` dependent __expf on a bounded value (a in (0,0.5])\n    // so it never overflows and the compiler can't hoist it. Measures MUFU.EX2 throughput, not memory.\n    float a = 0.25f + 1e-6f * (threadIdx.x + blockIdx.x);\n    for (int i = 0; i < iters; ++i) a = 0.5f * __expf(-fabsf(a));\n    out[blockIdx.x * blockDim.x + threadIdx.x] = a;\n}\ntorch::Tensor run_ex2(int64_t blocks, int64_t threads, int64_t iters) {\n    auto out = torch::empty({blocks * threads}, torch::dtype(torch::kFloat32).device(torch::kCUDA));\n    ex2_bench<<<blocks, threads>>>(out.data_ptr<float>(), (int)iters);\n    return out;\n}",
                  functions=['run_ex2'], extra_cuda_cflags=['-O3'], verbose=False)
from roofline.archs import get_arch
p = torch.cuda.get_device_properties(0)
blocks, threads, iters = p.multi_processor_count * 32, 256, 4096
total_exps = blocks * threads * iters
for _ in range(3): ex2.run_ex2(blocks, threads, iters); torch.cuda.synchronize()
s, e = torch.cuda.Event(True), torch.cuda.Event(True)
s.record()
for _ in range(20): ex2.run_ex2(blocks, threads, iters)
e.record(); torch.cuda.synchronize()
secs = s.elapsed_time(e) / 1e3 / 20
texps = total_exps / secs / 1e12
arch = get_arch(f'sm_{p.major}{p.minor}') if f'sm_{p.major}{p.minor}' in __import__('roofline.archs', fromlist=['ARCHS']).ARCHS else None
claim = (arch.exp_per_s/1e12) if (arch and arch.exp_per_s) else float('nan')
print(f'device {p.name} ({p.major}.{p.minor}) | {blocks} blocks x {threads} thr x {iters} iters')
print(f'achieved EX2: {texps:7.2f} TExp/s   |  roofline claim: {claim:.2f} TExp/s   |  ratio {texps/claim:.2f}x' if claim==claim else f'achieved EX2: {texps:.2f} TExp/s')
print('\n(Compute-bound microkernel -> the achievable EX2 peak. On B300 expect ~10.7; on B200 ~5.4 -> the 2x.)')

## 3. The softmax-term share of the decode floor (how much can 2×-exp move µs/tok?)

In [ ]:
# The roofline splits decode into t_mma / t_hbm / t_mufu. The 2x-exp only helps if t_mufu is a real
# share. For per-CTA-bound M=1 decode it is tiny -> the honest prediction: 2x-exp barely moves decode.
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_103')
print(f"{'shape':>22} | {'t_mma':>9} | {'t_hbm':>9} | {'t_mufu':>9} | {'mufu share':>10}")
for d in (128,):
    for N_k in (8192, 131072, 1048576):
        for G in (1, 8):
            e = estimate(arch, B=1, H=G*1, N_q=1, N_k=N_k, d=d, precision='nvfp4', G=G)
            tot = e.t_mma + e.t_hbm + e.t_mufu
            print(f'{f"d{d} Nk{N_k} G{G}":>22} | {e.t_mma*1e3:8.4f}ms | {e.t_hbm*1e3:8.4f}ms | '
                  f'{e.t_mufu*1e3:8.4f}ms | {e.t_mufu/tot*100:9.2f}%')
print('\nIf the mufu share is a few % -> 2x-exp halves a small term -> small decode win (the prediction).')
print('A larger share at long context would make the sm_103 exp lever matter — that is what we measure.')